## Lab 6: Working with different Model Formats (Keras, ONNX, TFLite and Quantization)
*Suggested time: 35-40 minutes*

**Use CPU for this lab**

In this lab we learn to train and save a model in a particular format. We also learn to work with different model formats by converting one format to another.

The goal of this lab is to show you:
- Train a Fully Connected Model and save it in TFLite format
- Train and convert a SK-Learn ML model to ONNX
- 8-bit fixed point Quantization
- How to view model graph using Netron

##### **Step 1:** Train and save a fully connected regression model (Fibonacci series) in Keras and TFLite format.



First run and understand the existing code. Then experiment by using these recommendations.
- Change the ***raw_seq*** to another series of your choice
- Tweak model architecture (layers or neurons) to see the model's loss reduces

In [0]:
import warnings
warnings.filterwarnings("ignore")

from numpy import array
from keras.models import Sequential
from keras.layers import Dense,Input
import tensorflow as tf
import os

# split a univariate sequence into samples
def split_sequence(sequence, n_steps_in, n_steps_out):
        X, y = list(), list()
        for i in range(len(sequence)):
                # find the end of this pattern
                end_index = i + n_steps_in
                out_end_index = end_index + n_steps_out
                # check if we are beyond the sequence
                if out_end_index > len(sequence):
                        break
                # gather input and output parts of the pattern
                seq_x, seq_y = sequence[i:end_index], sequence[end_index:out_end_index]
                X.append(seq_x)
                y.append(seq_y)
        return array(X), array(y)


#experimental Fibonaaci numbers
raw_seq = [0,1,1,2,3,5,8,13,21,34,55]

# choose a number of time steps for input and output
n_steps_in, n_steps_out = 3,2
# split into samples

X, y = split_sequence(raw_seq, n_steps_in, n_steps_out)

print ('Input sequence X')
print (X)
print ('Output sequence y',y)
print (y)
print ('Shape of input sequence:', X.shape)
print ('Shape of output sequence:', y.shape)

# define model
model = Sequential()
model.add(Input(shape=(n_steps_in,)))
model.add(Dense(25, activation='relu'))
model.add(Dense(25, activation='relu'))
model.add(Dense(n_steps_out))
print(model.summary())
model.compile(optimizer='adam', loss='mse')

# fit model
model.fit(X, y, epochs=50, verbose=1)

#Save model as contemporary .keras format
model.save('my_fibonacci.keras')

# Convert the model to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the TF Lite model.
with tf.io.gfile.GFile('fibonacci.tflite', 'wb') as f:
  f.write(tflite_model)

##### **Step 2:** Train and save K-means trained clustering model (Iris dataset) in ONNX format.



First run and understand the existing code. Then experiment by using these recommendations.
- Change the ***raw_seq*** to another series of your choice
- Tweak model architecture (layers or neurons) to see the model's loss reduces

In [0]:
!pip install -q skl2onnx

In [0]:
# Train a model.
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target
X = X.astype(np.float32)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42) # Added random_state for reproducibility
clr = RandomForestClassifier()
clr.fit(X_train, y_train)

# Calculate and print accuracy on the test set
y_pred = clr.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy on test set: {accuracy:.4f}")

# Convert into ONNX format.
from skl2onnx import to_onnx

onx = to_onnx(clr, X[:1])
with open("rf_iris.onnx", "wb") as f:
    f.write(onx.SerializeToString())

##### **Step 3:**

Get some details of the ONNX model. Try to understand [ONNX Versioning](https://onnx.ai/onnx/repo-docs/Versioning.html), and piece together the features of the generated graph - such as ONNX IR (Intermediate Representation) version, and number of nodes.

In [0]:
!pip install -q onnx

In [0]:
import onnx
import numpy as np

# Load the ONNX model from a file
model = onnx.load("rf_iris.onnx")

# Access top-level model properties
print(f"Producer Name: {model.producer_name}")
print(f"ONNX IR Version: {model.ir_version}")

# Access graph-level properties
graph = model.graph
print(f"\nGraph Name: {graph.name}")
print(f"Number of nodes: {len(graph.node)}")

##### **Step 4:** Demonstation of 8-bit Quantization.


- Typically used for model parameters (weights, activation functions)
- Calculation of scale and zero-point
    - scale = (max_val - min_val) / 255
    - zero_point = int(-min_val / scale)
- quantization_value (q) = round(x / scale) + zero_point
- dequantized = (q - zero_point) * scale

*First execute and understand the existing code. Then experiment by using these recommendations.*
- Change the list ***sample_values*** with values of your choice
- Update the min_val and max_val based on your list
- Experiment with 4-bit Quantization and see the error

In [0]:
import numpy as np

def quantize_and_dequantize(float_values, scale, zero_point):
    # Convert to numpy array for easier calculations
    np_float_values = np.array(float_values)

    # 1. Quantization:
    # Formula: q = round(x / scale) + zero_point
    # We also clamp the result to the 8-bit range [0, 255]
    quantized_values = np.round(np_float_values / scale) + zero_point
    quantized_values = np.clip(quantized_values, 0, 255).astype(np.uint8)

    # 2. Dequantization:
    # Formula: x_dequantized = (q - zero_point) * scale
    dequantized_values = (quantized_values.astype(np.float32) - zero_point) * scale

    # 3. Calculate Error
    error = dequantized_values - np_float_values

    return quantized_values.tolist(), dequantized_values.tolist(), error.tolist()

if __name__ == "__main__":
    # Define sample values
    sample_values = [-14.5, -6.1, 0.0, 5.7, 14.8]

    # Define scale and zero point
    # These are typically derived from the min/max of the data range.
    # For this example, let's assume a range of [-15, 15] which is mapped to [0, 255].
    min_val = -15
    max_val = 15
    scale = (max_val - min_val) / 255
    zero_point = int(-min_val / scale)

    print(f"Original float values: {sample_values}")
    print(f"Calculated Scale: {scale:.4f}")
    print(f"Calculated Zero Point: {zero_point}")
    print("\n--- Quantization and Dequantization Process ---")

    quantized, dequantized, errors = quantize_and_dequantize(sample_values, scale, zero_point)

    for i in range(len(sample_values)):
        print(f"Original: {sample_values[i]:.2f} -> Quantized: {quantized[i]:>3} -> Dequantized: {dequantized[i]:.4f} -> Error: {errors[i]:.4f}")

##### **Step 5:** Convert TFLite to ONNX.



In [0]:
!pip install -q tflite2onnx

In [0]:
import tflite2onnx

tflite_path = 'fibonacci.tflite' # Tflite model name/path
onnx_path = 'fibonacci.onnx' # ONNX model name/path
tflite2onnx.convert(tflite_path, onnx_path)

##### **Step 6:** View TFlite and ONNX model graphs using Netron

- Go to the Output section displayed on the right hand side panel
- Refresh directory contents by pressing the circular icon
- Locate and download the fibonnaci.onnx and fibonacci.tflite models to your local workstation
- Open Netron website (https://netron.app/) and upload the model that you wish to view

##### **Exercise:**  

Convert a ResNet-50 PyTorch model from PyTorch(.pt) to ONNX format.  View the converted model graph to ascertion shapes of input and output tensors. We will get you started by installing needed packages, downloading and converting the PyTorch ResNet-50 model to ONNX.
*   Access the ResNet-50 based Classification PyTorch model.
*  Convert the downloaded ResNet-50 PyTorch model to ONNX.


Now, use Netron to view the converted ONNX ResNet model's input and output tensor shapes.



In [0]:
!pip install -q onnx onnxscript

##### Download sample ResNet-50 model trained using PyTorch

In [0]:
import torch
from torchvision.models import resnet50, ResNet50_Weights

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

##### Convert model from PyTorch to ONNX format

In [0]:
model.eval()

sample_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(model,          # model being run
  sample_input,                    # model input (or a tuple for multiple inputs)
  "./ImageClassifier.onnx",       # where to save the model
  export_params=True,             # store the trained parameter weights inside the model file
  opset_version=20,               # the ONNX version to export the model to
  do_constant_folding=True,       # whether to execute constant folding for optimization
  input_names = ['modelInput'],   # the model's input names
  output_names = ['modelOutput'], # the model's output names
)

print('Model has been converted to ONNX')

##### Exercise

Display and check the converted ImageClassifier.onnx file using Netron. Note this model has been trained on the ImageNet dataset.

##### Solution

<details>
    <summary> Click here for our answer </summary>

    - Install Netron or use the Netron website to view the graph of the converted PyTorch ResNet50 (ImageClassifier.onnx) model. Check to ensure that the output tensor mentions 1000 classes.

   
</details>






